In [ ]:
import os
from google.colab import drive
os.chdir('/content')
drive.mount('/content/drive', force_remount=True)

!pip install -q git+https://github.com/compphoto/Intrinsic.git
!pip install opencv-python
os.chdir('/content/drive/MyDrive/3dgs_decomposition')
import matplotlib.pyplot as plt
import numpy as np

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# fetch data and run albedo predictor
from data.get_data import get_data_for_predictors
from predictors.colorful_iid import colorful_iid_predict
from predictors.marigold import marigold_predict
from predictors.rgbx.rgbx import rgbx_predict

scenes = [('hypersim', 'ai_001_001'), ('hypersim', 'ai_002_001'), ('mipnerf360', 'bicycle'), ('mipnerf360', 'garden'), ('mipnerf360', 'room')]


loading v2 weights


Using cache found in /root/.cache/torch/hub/facebookresearch_WSL-Images_main
Using cache found in /root/.cache/torch/hub/rwightman_gen-efficientnet-pytorch_master
Using cache found in /root/.cache/torch/hub/facebookresearch_WSL-Images_main
Using cache found in /root/.cache/torch/hub/facebookresearch_WSL-Images_main
Using cache found in /root/.cache/torch/hub/facebookresearch_WSL-Images_main


In [ ]:
# code to generate 2D priors with rgbx and save them to drive for future use
from PIL import Image

for dataset, scene in scenes:
  images = get_data_for_predictors(dataset=dataset, scene=scene)[5:] # Modified since we've already done the first five images 
  albedos, diffuse_shading, residuals = rgbx_predict(images, cache_dir='/content/model_cache')


  os.makedirs(f'rgbx/{dataset}/{scene}/albedos', exist_ok=True)
  os.makedirs(f'rgbx/{dataset}/{scene}/shading', exist_ok=True)
  os.makedirs(f'rgbx/{dataset}/{scene}/residuals', exist_ok=True)

  for idx, image in enumerate(images):
    Image.fromarray(albedos[idx]).save(f'rgbx/{dataset}/{scene}/albedos/{os.path.splitext(os.path.basename(image))[0]}.jpg')
    Image.fromarray(diffuse_shading[idx]).save(f'rgbx/{dataset}/{scene}/shading/{os.path.splitext(os.path.basename(image))[0]}.jpg')
    Image.fromarray(residuals[idx]).save(f'rgbx/{dataset}/{scene}/residuals/{os.path.splitext(os.path.basename(image))[0]}.jpg')
    np.save(f'rgbx/{dataset}/{scene}/albedos/{os.path.splitext(os.path.basename(image))[0]}.npy', albedos[idx])
    np.save(f'rgbx/{dataset}/{scene}/shading/{os.path.splitext(os.path.basename(image))[0]}.npy', diffuse_shading[idx])
    np.save(f'rgbx/{dataset}/{scene}/residuals/{os.path.splitext(os.path.basename(image))[0]}.npy', residuals[idx])